# 12个数据文件夹批量特征提取（UTF-8）

该 notebook 用于对 `data/dataset_build_260505` 下指定文件夹进行特征提取，并按文件夹导出 CSV。

- 文件编码：UTF-8（notebook 默认按 UTF-8 保存）
- 每行对应一个样本
- 最后 3 列固定为：`label`、`sample_name`、`sample_rate`
- 特征按频带展开，列名前缀为频带名（例如 `b_1k_10k__SC_mean`）


In [7]:
from __future__ import annotations

from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor, as_completed
from dataclasses import replace
from pathlib import Path
import sys
from typing import Iterable

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

workspace = Path.cwd()
if not (workspace / 'src').exists():
    workspace = workspace.parent
if str(workspace / 'src') not in sys.path:
    sys.path.insert(0, str(workspace / 'src'))

from fea_cpt.base import FeatureRecord
from fea_cpt.features import compute_all_features
from fea_cpt.io_utils import load_feature_record
from fea_cpt.params import DEFAULT_FEATURE_PARAMS
from fea_cpt.signal_ops import build_context, butter_filter

print(f'workspace = {workspace}')


workspace = e:\codes\ZZ-BK


In [8]:
# =========================
# Global configuration
# =========================
DATASET_ROOT = workspace / 'data' / 'dataset_build_260505'
OUTPUT_ROOT = workspace / 'outputs' / 'dataset_build_260505_features'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# Dataset folder -> binary label
DATASET_LABELS = {
    'BK14_f130_test': 1,
    'BK14_f130_train': 1,
    'BK14_f130a_test': 1,
    'BK14_f130a_train': 1,
    'BK14_f130c_test': 1,
    'BK14_f130c_train': 1,
    'F130_te': 0,
    'F130_tr': 0,
    'F130A_te': 0,
    'F130A_tr': 0,
    'F130C_te': 0,
    'F130C_tr': 0,
}

# Frequency bands in Hz
BANDS = [
    ('b_1k_100k', (1_000.0, 100_000.0)),
    ('b_1k_10k', (1_000.0, 10_000.0)),
    ('b_10k_20k', (10_000.0, 20_000.0)),
    ('b_20k_40k', (20_000.0, 40_000.0)),
    ('b_40k_60k', (40_000.0, 60_000.0)),
    ('b_10k_60k', (10_000.0, 60_000.0)),
    ('b_60k_100k', (60_000.0, 100_000.0)),
]

# Feature selection mode: 'all' / 'top_n' / 'manual'
FEATURE_MODE = 'all'
TOP_N = 50
MANUAL_FEATURES: list[str] = []

# Preprocess band: de-mean + 1k~100k band-pass
PREPROC_BAND = (1_000.0, 100_000.0)

# None means full folder; int means first N samples per folder
MAX_SAMPLES_PER_FOLDER: int | None = 100

# Parallel settings
N_WORKERS = 12
PARALLEL_BACKEND = 'thread'  # 'thread' or 'process'

# Output CSV name template
CSV_NAME_TEMPLATE = '{folder_name}_features.csv'

print(f'DATASET_ROOT = {DATASET_ROOT}')
print(f'OUTPUT_ROOT  = {OUTPUT_ROOT}')
print(f'MAX_SAMPLES_PER_FOLDER = {MAX_SAMPLES_PER_FOLDER}')
print(f'N_WORKERS = {N_WORKERS}, PARALLEL_BACKEND={PARALLEL_BACKEND}')


DATASET_ROOT = e:\codes\ZZ-BK\data\dataset_build_260505
OUTPUT_ROOT  = e:\codes\ZZ-BK\outputs\dataset_build_260505_features
MAX_SAMPLES_PER_FOLDER = 100
N_WORKERS = 12, PARALLEL_BACKEND=thread


In [9]:
# =========================
# Functions: preprocess / params / extraction pipeline
# =========================

_PARAMS_CACHE: dict[tuple[float, tuple[float, float]], object] = {}


def preprocess_signal(raw_signal: np.ndarray, sample_rate: float, band: tuple[float, float]) -> np.ndarray:
    """De-mean + Butterworth band-pass."""
    demeaned = np.asarray(raw_signal, dtype=float) - float(np.mean(raw_signal))
    return butter_filter(demeaned, sample_rate=sample_rate, band_hz=band, order=4)


def _safe_band(low: float, high: float, nyq: float) -> tuple[float, float]:
    low = max(1.0, min(low, nyq * 0.98))
    high = max(low + 1.0, min(high, nyq * 0.995))
    return (float(low), float(high))


def build_params_for_band(band: tuple[float, float], sample_rate: float):
    """Build feature parameters for one target band."""
    low, high = band
    nyq = sample_rate / 2.0
    low, high = _safe_band(low, high, nyq)
    span = max(high - low, 10.0)

    low_band = _safe_band(low, low + 0.30 * span, nyq)
    mid_band = _safe_band(low + 0.30 * span, low + 0.60 * span, nyq)
    high1_band = _safe_band(low + 0.50 * span, low + 0.80 * span, nyq)
    high2_band = _safe_band(low + 0.60 * span, high, nyq)
    harmonic_band = _safe_band(low + 0.50 * span, high, nyq)

    ridge_main = _safe_band(low, low + 0.65 * span, nyq)
    ridge_h2 = _safe_band(max(low * 2.0, low + 0.20 * span), min(high * 2.0, nyq * 0.995), nyq)

    return replace(
        DEFAULT_FEATURE_PARAMS,
        highpass_hz=1_000.0,
        main_band_hz=(low, high),
        low_band_hz=low_band,
        mid_band_hz=mid_band,
        high1_band_hz=high1_band,
        high2_band_hz=high2_band,
        harmonic_band_hz=harmonic_band,
        ridge_main_search_hz=ridge_main,
        ridge_h2_search_hz=ridge_h2,
        n_jobs=1,
    )


def get_params_for_band(band: tuple[float, float], sample_rate: float):
    key = (float(sample_rate), (float(band[0]), float(band[1])))
    params = _PARAMS_CACHE.get(key)
    if params is None:
        params = build_params_for_band(band, sample_rate)
        _PARAMS_CACHE[key] = params
    return params


def choose_feature_names(rows: list[dict[str, float | int | str]]) -> list[str]:
    """Select feature columns by FEATURE_MODE."""
    all_cols = [c for c in rows[0].keys() if c not in {'label', 'sample_name', 'sample_rate', 'folder_name'}]
    if FEATURE_MODE == 'all':
        return all_cols
    if FEATURE_MODE == 'manual':
        return [c for c in all_cols if c in set(MANUAL_FEATURES)]
    if FEATURE_MODE == 'top_n':
        scored = []
        for c in all_cols:
            arr = pd.to_numeric(pd.Series([r.get(c, np.nan) for r in rows]), errors='coerce').fillna(0.0)
            scored.append((c, float(arr.std())))
        scored.sort(key=lambda x: x[1], reverse=True)
        return [name for name, _ in scored[:TOP_N]]
    raise ValueError(f'Unsupported FEATURE_MODE: {FEATURE_MODE}')


def list_npz_files(folder: Path) -> list[Path]:
    return sorted(folder.glob('*.npz'))


def extract_one_sample(path: Path, label: int, band_defs: Iterable[tuple[str, tuple[float, float]]], folder_name: str) -> dict[str, float | int | str]:
    """Extract all band features for one sample and return one row dict."""
    record = load_feature_record(path)
    filtered = preprocess_signal(record.signal, record.sample_rate, PREPROC_BAND)
    proc_record = FeatureRecord(
        sample_id=record.sample_id,
        sample_name=record.sample_name,
        sample_type=record.sample_type,
        sample_type_code=record.sample_type_code,
        path=record.path,
        signal=filtered,
        sample_rate=record.sample_rate,
        metadata=record.metadata,
    )

    row: dict[str, float | int | str] = {}
    for band_name, band in band_defs:
        params = get_params_for_band(band, sample_rate=proc_record.sample_rate)
        context = build_context(proc_record, params)
        result = compute_all_features(context)
        for k, v in result.features.items():
            row[f'{band_name}__{k}'] = float(v)

    row['label'] = int(label)
    row['sample_name'] = proc_record.sample_name
    row['sample_rate'] = float(proc_record.sample_rate)
    row['folder_name'] = folder_name
    return row


def _executor_cls() -> type[ThreadPoolExecutor | ProcessPoolExecutor]:
    if str(PARALLEL_BACKEND).lower() == 'process':
        return ProcessPoolExecutor
    return ThreadPoolExecutor


def run_one_folder(folder_name: str, label: int) -> pd.DataFrame:
    """Process one folder and export one CSV."""
    folder = DATASET_ROOT / folder_name
    files = list_npz_files(folder)
    if not files:
        raise FileNotFoundError(f'No npz files found: {folder}')

    if MAX_SAMPLES_PER_FOLDER is not None:
        files = files[:MAX_SAMPLES_PER_FOLDER]

    rows: list[dict[str, float | int | str]] = []
    max_workers = max(1, min(int(N_WORKERS), len(files)))
    progress_desc = f'{folder_name} ({len(files)} samples)'
    band_defs = tuple(BANDS)

    executor_cls = _executor_cls()
    try:
        with executor_cls(max_workers=max_workers) as ex:
            futures = [ex.submit(extract_one_sample, fp, label, band_defs, folder_name) for fp in files]
            with tqdm(total=len(futures), desc=progress_desc, leave=True) as pbar:
                for fut in as_completed(futures):
                    rows.append(fut.result())
                    pbar.update(1)
    except Exception as exc:
        if executor_cls is ProcessPoolExecutor:
            print(f'[WARN] ProcessPool failed; fallback to ThreadPool: {exc}')
            with ThreadPoolExecutor(max_workers=max_workers) as ex:
                futures = [ex.submit(extract_one_sample, fp, label, band_defs, folder_name) for fp in files]
                with tqdm(total=len(futures), desc=progress_desc, leave=True) as pbar:
                    for fut in as_completed(futures):
                        rows.append(fut.result())
                        pbar.update(1)
        else:
            raise

    selected_features = choose_feature_names(rows)
    final_cols = selected_features + ['label', 'sample_name', 'sample_rate', 'folder_name']
    frame = pd.DataFrame(rows)
    frame = frame.reindex(columns=final_cols)

    out_csv = OUTPUT_ROOT / CSV_NAME_TEMPLATE.format(folder_name=folder_name)
    frame.to_csv(out_csv, index=False, encoding='utf-8-sig')
    print(f'[OK] {folder_name}: {len(frame)} samples -> {out_csv}')
    return frame


In [ ]:
# =========================
# Run batch extraction and summarize
# =========================
summary = []
for folder_name, label in DATASET_LABELS.items():
    df_folder = run_one_folder(folder_name, label)
    summary.append({
        'folder': folder_name,
        'label': label,
        'n_samples': len(df_folder),
        'n_columns': df_folder.shape[1],
    })

summary_df = pd.DataFrame(summary).sort_values('folder').reset_index(drop=True)
summary_df


BK14_f130_test (100 samples):  32%|███▏      | 32/100 [04:44<10:04,  8.89s/it] 


In [ ]:
# Preview one exported CSV
preview_csv = OUTPUT_ROOT / CSV_NAME_TEMPLATE.format(folder_name='BK14_f130_test')
preview_df = pd.read_csv(preview_csv)
print(preview_df.shape)
preview_df.head(3)
